Step 0 - Setup

In [7]:
!pip install -q pypdf tiktoken langchain-text-splitters google-genai chromadb ipywidgets tqdm python-dotenv

import os

GEMINI_API_KEY = None
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print("API Key cargada desde Colab userdata.")
except Exception:
    pass

if not GEMINI_API_KEY:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')

if not GEMINI_API_KEY:
    raise EnvironmentError("GEMINI_API_KEY no encontrada.")

os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
print(f"API Key configurada: {GEMINI_API_KEY[:8]}...")

API Key cargada desde Colab userdata.
API Key configurada: AIzaSyBe...


Step 1 - PDF Text Extraction

In [11]:
import re
from pypdf import PdfReader

PDF_PATH = "/content/beca18_reglamento.pdf"

def clean_text(text):
    text = re.sub(r'\r\n|\r', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n\s*\n', '\n\n', text)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    return text.strip()

reader = PdfReader(PDF_PATH)
pages_text = []
for i, page in enumerate(reader.pages, start=1):
    raw = page.extract_text() or ""
    pages_text.append(f"[PAGE {i}]\n{clean_text(raw)}")

full_text = "\n\n".join(pages_text)
print(f"Páginas: {len(pages_text)} | Chars: {len(full_text):,} | Palabras: {len(full_text.split()):,}")

Páginas: 138 | Chars: 373,198 | Palabras: 55,202


Step 2 — Tokenization and Chunking Justification

In [12]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

enc = tiktoken.get_encoding("cl100k_base")
print(f"Total tokens: {len(enc.encode(full_text)):,}")

def token_length(text): return len(enc.encode(text))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400, chunk_overlap=60,
    length_function=token_length,
    separators=["\n\n", "\n", ". ", " ", ""],
)
raw_chunks = splitter.split_text(full_text)

def extract_page(text):
    m = re.findall(r'\[PAGE (\d+)\]', text)
    return int(m[-1]) if m else -1

chunks = [{"id": f"chunk_{i:04d}", "text": t,
           "metadata": {"document": "Reglamento Beca 18", "topic": "beca18",
                        "language": "es", "page": extract_page(t), "chunk_id": i}}
          for i, t in enumerate(raw_chunks)]
print(f"Chunks: {len(chunks)}")

Total tokens: 109,724
Chunks: 374


Step 3 — Embeddings

In [17]:
import time, random
from google import genai
from google.genai import types as genai_types
from tqdm.notebook import tqdm

EMBEDDING_MODEL, EMBEDDING_DIM = "gemini-embedding-001", 768
client = genai.Client(api_key=GEMINI_API_KEY)

def embed_with_backoff(texts, task_type="RETRIEVAL_DOCUMENT", max_retries=6):
    for attempt in range(max_retries):
        try:
            r = client.models.embed_content(
                model=EMBEDDING_MODEL, contents=texts,
                config=genai_types.EmbedContentConfig(
                    task_type=task_type, output_dimensionality=EMBEDDING_DIM))
            return [e.values for e in r.embeddings]
        except Exception as e:
            if any(x in str(e).lower() for x in ['quota','rate','429']):
                wait = (2**attempt) + random.uniform(0,1)
                print(f"Rate limit, esperando {wait:.1f}s...")
                time.sleep(wait)
            else: raise
    raise RuntimeError("Max retries alcanzado.")

def embed_documents(texts, batch_size=5):
    results = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        results.extend(embed_with_backoff(batch, "RETRIEVAL_DOCUMENT"))
        time.sleep(5)  # espera 5 seg entre batches
    return results

def embed_query(query):
    return embed_with_backoff([query], task_type="RETRIEVAL_QUERY")[0]

Step 4 — Vector Database

In [18]:
import chromadb
from chromadb.config import Settings

chroma_client = chromadb.PersistentClient(
    path="chroma_db_beca18",
    settings=Settings(anonymized_telemetry=False)
)

collection = chroma_client.get_or_create_collection(
    name="beca18_reglamento",
    metadata={"hnsw:space": "cosine"}
)

if collection.count() == 0:
    embeddings = embed_documents([c['text'] for c in chunks])
    for start in range(0, len(chunks), 100):  # 👈 indentado dentro del if
        sl = slice(start, start+100)
        collection.add(
            ids=[c['id'] for c in chunks[sl]],
            embeddings=embeddings[sl],
            documents=[c['text'] for c in chunks[sl]],
            metadatas=[c['metadata'] for c in chunks[sl]]
        )
    print(f"Indexados {collection.count()} documentos.")
else:
    print(f"Ya indexado: {collection.count()} documentos (idempotente).")

  0%|          | 0/75 [00:00<?, ?it/s]

Indexados 374 documentos.


Step 5 — Semantic Search

In [19]:
def semantic_search(query, k=3):
    results = collection.query(
        query_embeddings=[embed_query(query)], n_results=k,
        include=["documents","metadatas","distances"])
    return [{"text": t, "metadata": m, "distance": round(float(d),4)}
            for t,m,d in zip(results["documents"][0],
                             results["metadatas"][0],
                             results["distances"][0])]

# Test top-3
for i, r in enumerate(semantic_search("requisitos para postular a Beca 18", k=3), 1):
    print(f"[{i}] Pág {r['metadata']['page']} | dist={r['distance']}\n{r['text'][:300]}\n")

[1] Pág -1 | dist=0.1869
*En el caso de la Beca 18 
Ordinaria el postulante debe 
haber egresado de la 
Educación Básica Regular 
(EBR) o Básica Alternativa 
(EBA), como máximo en los 
tres (3) años anteriores al año 
de la publicación de las Bases 
(entre 2023 y 2025), con 
excepción de las personas que 
acrediten discapac

[2] Pág -1 | dist=0.1988
Los estudiantes o egresados de EBA que 
transitaron de EBR a EBA en el penúltimo grado 
académico (3ro o 4to EBR), pueden registrar las 
competencias y notas correspondientes al 
grado académico cursado en el colegio EBR, en 
la sección de 2do o 3ero EBA, según 
corresponda, en el Módulo de Postulac

[3] Pág -1 | dist=0.2004
El Concurso Beca 18 y Becas Especiales - Convocatoria 2026, es realizado bajo 
principios de meritocracia , equidad y eficiencia en el uso de los recursos 
públicos, contribuyendo al desarrollo nacional y regional, de conformidad con lo 
establecido en las Bases. 

Artículo 2.- Objetivos de las Base



Step 6 — Grounded Generation

In [21]:
GENERATION_MODEL = "gemini-2.5-flash"
SYSTEM_PROMPT = """Eres un asistente especializado en el Reglamento de Beca 18 (PRONABEC, Perú).
INSTRUCCIONES ESTRICTAS:
1. Responde ÚNICAMENTE usando el contexto proporcionado.
2. NO inventes información ni uses conocimiento externo.
3. Cita páginas con [Página N].
4. Si no hay información, responde: "No encuentro información sobre ese tema en el Reglamento de Beca 18 proporcionado."
5. Presenta listas cuando haya múltiples ítems."""

def answer_with_context(question, k=5):
    sources = semantic_search(question, k=k)
    context = "\n\n".join(
        f"[FRAGMENTO {i} - Página {r['metadata'].get('page')}]\n{r['text']}"
        for i,r in enumerate(sources,1))
    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=f"Contexto:\n\n{context}\n\n---\nPregunta: {question}",
        config=genai_types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT, temperature=0.1, max_output_tokens=1024))
    return {"answer": response.text, "sources": sources}

for i, q in enumerate([
    "¿Cuáles son los requisitos para postular a Beca 18?",
    "¿Cuáles son las modalidades de Beca 18?",
    "¿Cuál es el monto de la subvención mensual?",
    "¿Cuáles son las obligaciones del becario?",
    "¿En qué casos se puede perder la beca?",
    "¿Cuál es la receta para preparar ceviche peruano?",   # off-topic
], 1):
    r = answer_with_context(q, k=4)
    print(f"{'='*60}\nPREGUNTA {i}: {q}\n{r['answer']}\n")

PREGUNTA 1: ¿Cuáles son los requisitos para postular a Beca 18?
Para postular a Beca 18, se requiere lo siguiente:

*   **Para Beca 18 Ordinaria:**
    *   Haber egresado de la

PREGUNTA 2: ¿Cuáles son las modalidades de Beca 18?
Las modalidades de Beca 18 son las siguientes:

*   Beca 18 Ordinaria [Página -1, Página 89, Página 3]
*   Beca de Formación en Educación Intercultural Bilingüe - Beca EIB [Página -1, Página 89, Página 3]
*   Beca para Adolescentes con Protección Estatal - Beca Protección [Página -1, Página 89, Página 3]
*   Beca para Comunidades Nativas Amazónicas – Beca CNA [Página -1, Página 89, Página 3]
*   Beca para Licenciados del Servicio Militar Voluntario - Beca FF.AA. [Página -1, Página 89, Página 3]
*   Beca para pobladores del Valle de los ríos Apurímac, Ene y Mantaro - Beca VRAEM [Página -1, Página 89, Página 3]
*   Beca para pobladores residentes en el Huallaga - Beca Huallaga [Página -1, Página 89, Página 3]
*   Beca para Pueblo Afroperuano - Beca PA [Página -1

Step 7 — Interactive Chat Interface

In [22]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

question_input = widgets.Text(placeholder="Escribe tu pregunta...", layout=widgets.Layout(width="72%", height="36px"))
ask_button     = widgets.Button(description="Ask",   button_style="primary", icon="search", layout=widgets.Layout(width="80px", height="36px"))
clear_button   = widgets.Button(description="Clear", button_style="warning", icon="trash",  layout=widgets.Layout(width="80px", height="36px"))
k_slider       = widgets.IntSlider(value=4, min=1, max=10, description="Chunks (k):", style={"description_width":"90px"}, layout=widgets.Layout(width="350px"))
status_label   = widgets.HTML(value="")
answer_output  = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="12px", min_height="80px"))
sources_output = widgets.Output()
accordion      = widgets.Accordion(children=[sources_output])
accordion.set_title(0, "Fuentes recuperadas")
accordion.selected_index = None

def on_ask(b):
    q = question_input.value.strip()
    if not q: return
    status_label.value = "<span style='color:#1a73e8'>Generando...</span>"
    ask_button.disabled = True
    try:
        result = answer_with_context(q, k=k_slider.value)
        with answer_output:
            clear_output(wait=True)
            display(HTML(f"<b>Pregunta:</b> {q}<hr><div style='white-space:pre-wrap'>{result['answer']}</div>"))
        with sources_output:
            clear_output(wait=True)
            html = "".join(
                f"<div style='margin:6px 0;padding:8px;background:#f5f5f5;border-radius:4px;font-size:.85em'>"
                f"<b>Fragmento {i}</b> | Pág <b>{s['metadata'].get('page')}</b> | dist <b>{s['distance']}</b>"
                f"<div style='margin-top:4px'>{s['text'][:280]}...</div></div>"
                for i,s in enumerate(result['sources'],1))
            display(HTML(html))
        accordion.set_title(0, f"Fuentes recuperadas ({len(result['sources'])} fragmentos)")
        status_label.value = "<span style='color:green'>Listo.</span>"
    except Exception as e:
        status_label.value = f"<span style='color:red'>Error: {e}</span>"
    finally:
        ask_button.disabled = False

def on_clear(b):
    question_input.value = ""; status_label.value = ""
    with answer_output: clear_output()
    with sources_output: clear_output()
    accordion.set_title(0, "Fuentes recuperadas"); accordion.selected_index = None

ask_button.on_click(on_ask)
clear_button.on_click(on_clear)
question_input.on_submit(lambda w: on_ask(None))

display(widgets.VBox([
    widgets.HTML("<h2 style='color:#1a73e8'>Chatbot Beca 18</h2><p style='color:#555'>Reglamento PRONABEC</p>"),
    widgets.HBox([question_input, ask_button, clear_button]),
    widgets.HBox([k_slider]),
    status_label, answer_output, accordion
], layout=widgets.Layout(padding="16px", width="100%")))